## GIZA 

In [ ]:
import pandas as pd
giza_oov = pd.read_csv('../ebible-corpus/dhao/ot_words_confidence_score.csv')
dict = pd.read_csv('../dictionary/dhao_ind_dictionary.csv')

In [16]:
import spacy
nlp = spacy.blank("id")
nlp.add_pipe("lemmatizer", config={"mode": "lookup"})
nlp.initialize()

# Normalize function for matching (replaces dashes with spaces)
def normalize_for_matching(text):
    return (text.str.lower()
            .str.replace(r'-', ' ', regex=True)  # Replace dashes with spaces for matching
            .str.replace(r'[0-9]', '', regex=True)  # Remove numerical digits
            .str.replace(r'[^\w\s]', '', regex=True)  # Remove punctuation (except word chars and spaces)
            .str.strip()  # Remove leading/trailing whitespace
            .str.replace(r'\s+', ' ', regex=True))  # Replace multiple spaces with single space
    
def normalize_source_for_matching(text_series):
    # Helper function to lemmatize individual text
    def lemmatize_text(text):
        doc = nlp(text)
        return [token.lemma_.lower() for token in doc][0] if doc else text.lower()
    
    # Apply lemmatization to each item in the series
    lemmatized_series = text_series.apply(lemmatize_text)
    
    # Apply normalization using pandas .str methods
    return (lemmatized_series.str.lower()
            .str.replace(r'-', ' ', regex=True)  # Replace dashes with spaces for matching
            .str.replace(r'[0-9]', '', regex=True)  # Remove numerical digits
            .str.replace(r'[^\w\s]', '', regex=True)  # Remove punctuation (except word chars and spaces)
            .str.strip()  # Remove leading/trailing whitespace
            .str.replace(r'\s+', ' ', regex=True))  # Replace multiple spaces with single space

# Normalize function for final display (keeps dashes)
def normalize_for_display(text):
    return (text.str.lower()
            .str.replace(r'[0-9]', '', regex=True)  # Remove numerical digits
            .str.replace(r'[^\w\s-]', '', regex=True)  # Remove punctuation but keep dashes and word chars/spaces
            .str.strip()  # Remove leading/trailing whitespace
            .str.replace(r'\s+', ' ', regex=True))  # Replace multiple spaces with single space

In [18]:

N = 100
# Create temporary normalized versions for matching
all_oov_for_merge = giza_oov[["source_word", "confidence_score"]].copy()  # Keep confidence_score for ranking
all_oov_for_merge["source_word_normalized"] = normalize_source_for_matching(all_oov_for_merge["source_word"])
all_oov_for_merge["source_word_display"] = normalize_for_display(all_oov_for_merge["source_word"])
# Drop original source_word to avoid _x, _y suffixes
all_oov_for_merge = all_oov_for_merge.drop(columns=["source_word"])

dict_for_merge = dict.copy()
dict_for_merge["source_word_normalized"] = normalize_for_matching(dict_for_merge["source_word"])
# Drop original source_word to avoid _x, _y suffixes
dict_for_merge = dict_for_merge.drop(columns=["source_word"])

# Perform the merge using normalized versions
oov_in_dict = pd.merge(
    all_oov_for_merge, 
    dict_for_merge, 
    left_on="source_word_normalized",
    right_on="source_word_normalized", 
    how="inner"
)

# Use the display version (normalized but with dashes) for final output
oov_in_dict["source_word"] = oov_in_dict["source_word_display"]
oov_in_dict = oov_in_dict.drop(columns=["source_word_normalized", "source_word_display"])

print(f"Total matches found: {len(oov_in_dict)}")
print(f"Unique source_words found: {oov_in_dict['source_word'].nunique()}")

# Step 1: Identify top N unique source_words based on confidence_score
# For each unique source_word, find the lowest confidence_score
min_confidence_per_word = (oov_in_dict
                          .groupby('source_word')['confidence_score']
                          .min()  # Get minimum confidence_score for each source_word
                          .reset_index()
                          .sort_values('confidence_score')  # Sort by confidence_score
                          .head(N))  # Take top N unique source_words

top_n_source_words = min_confidence_per_word['source_word'].tolist()
print(f"\nSelected top {N} unique source_words (with lowest confidence scores)")

# Step 2: Keep ALL entries for those top N unique source_words
final_result = (oov_in_dict[oov_in_dict['source_word'].isin(top_n_source_words)]
                .sort_values(['confidence_score', 'source_word']))  # Sort by confidence_score, then source_word

print(f"Final result has {len(final_result)} total entries for {final_result['source_word'].nunique()} unique source_words")

# Show breakdown of entries per source_word
entries_per_word = final_result['source_word'].value_counts().sort_values(ascending=False)
print(f"Entries per source_word - Max: {entries_per_word.max()}, Min: {entries_per_word.min()}, Avg: {entries_per_word.mean():.1f}")
if entries_per_word.max() > 1:
    print(f"Source_words with multiple entries: {(entries_per_word > 1).sum()}")
    print("Top 5 source_words with most entries:")
    display(entries_per_word.head().to_frame('entry_count'))

# Reorder columns to put source_word first, followed by target_word
columns = ["source_word", "target_word"] + [col for col in final_result.columns if col not in ["source_word", "target_word"]]
final_result = final_result[columns]

final_result

Total matches found: 2351
Unique source_words found: 1188

Selected top 100 unique source_words (with lowest confidence scores)
Final result has 219 total entries for 100 unique source_words
Entries per source_word - Max: 12, Min: 1, Avg: 2.2
Source_words with multiple entries: 57
Top 5 source_words with most entries:


,entry_count
source_word,
kami,12
itu,9
kesalahan,6
semua,6
suka,6


,source_word,target_word,confidence_score,pos,page_number
0,melawan,lab'a,2.544016e-44,verb,18
1,melawan,sisu,2.544016e-44,verb,18
2,memohon,aj'u,2.544016e-44,noun,26
3,semua,mèu-mèu,2.603207e-44,adverb,30
4,semua,aa'i,2.603207e-44,quantifier,30
...,...,...,...,...,...
2017,tongkat,tatea,6.449826e-01,noun,35
2096,biarkan,hia,6.836618e-01,verb,5
2097,biarkan,ai-hèb'a,6.836618e-01,idiom,5
2153,makanan,nganga'a,7.218543e-01,noun,19


In [19]:
final_result.to_csv("/data/projects/punim0478/setiawand/bible-nmt/dictionary/giza_100_oov_dict.csv", index=False)

# Top Freq 100 OOV

Get top 100 most frequent oov in OT exlcuding NER.

In [21]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline
import torch


freq_oov = pd.read_csv("/data/projects/punim0478/setiawand/bible-nmt/ebible-corpus/dhao/ot_oov_freq_count.csv")
dict = pd.read_csv('../dictionary/dhao_ind_dictionary.csv')

processed_count = 0

# Now apply the same logic as the confidence score version but for frequency count
# Create temporary normalized versions for matching (similar to confidence score version)
freq_oov_for_merge = freq_oov[["word", "freq_count"]].copy()  # Keep freq_count for ranking

freq_oov_for_merge["word_normalized"] = normalize_source_for_matching(freq_oov_for_merge["word"])
freq_oov_for_merge["word_display"] = normalize_for_display(freq_oov_for_merge["word"])
# Drop original word to avoid _x, _y suffixes
freq_oov_for_merge = freq_oov_for_merge.drop(columns=["word"])

dict_for_merge = dict.copy()
dict_for_merge["source_word_normalized"] = normalize_for_matching(dict_for_merge["source_word"])
# Drop original source_word to avoid _x, _y suffixes
dict_for_merge = dict_for_merge.drop(columns=["source_word"])

# Perform the merge using normalized versions
freq_oov_in_dict = pd.merge(
    freq_oov_for_merge, 
    dict_for_merge, 
    left_on="word_normalized",
    right_on="source_word_normalized", 
    how="inner"
)

# Use the display version (normalized but with dashes) for final output
freq_oov_in_dict["word"] = freq_oov_in_dict["word_display"]
freq_oov_in_dict = freq_oov_in_dict.drop(columns=["word_normalized", "source_word_normalized", "word_display"])

print(f"Total matches found: {len(freq_oov_in_dict)}")
print(f"Unique words found: {freq_oov_in_dict['word'].nunique()}")

# Step 1: Identify top 100 unique words based on freq_count (HIGHEST frequency, not lowest like confidence)
# For each unique word, find the highest freq_count
max_freq_per_word = (freq_oov_in_dict
                    .groupby('word')['freq_count']
                    .max()  # Get maximum freq_count for each word
                    .reset_index()
                    .sort_values('freq_count', ascending=False)  # Sort by freq_count descending (highest first)
                    .head(100))  # Take top 100 unique words

top_100_words = max_freq_per_word['word'].tolist()
print(f"\nSelected top 100 unique words (with highest frequency counts)")

# Step 2: Keep ALL entries for those top 100 unique words
final_freq_result = (freq_oov_in_dict[freq_oov_in_dict['word'].isin(top_100_words)]
                    .sort_values(['freq_count', 'word'], ascending=[False, True]))  # Sort by freq_count descending, then word

print(f"Final result has {len(final_freq_result)} total entries for {final_freq_result['word'].nunique()} unique words")

# Show breakdown of entries per word
entries_per_word = final_freq_result['word'].value_counts().sort_values(ascending=False)
print(f"Entries per word - Max: {entries_per_word.max()}, Min: {entries_per_word.min()}, Avg: {entries_per_word.mean():.1f}")
if entries_per_word.max() > 1:
    print(f"Words with multiple entries: {(entries_per_word > 1).sum()}")
    print("Top 5 words with most entries:")
    display(entries_per_word.head().to_frame('entry_count'))

# Reorder columns to put word first, followed by target_word
columns = ["word", "target_word"] + [col for col in final_freq_result.columns if col not in ["word", "target_word"]]
# Rename 'word' column to 'source_word' before reordering
final_freq_result = final_freq_result[columns]
final_freq_result = final_freq_result.rename(columns={"word": "source_word"})

final_freq_result.to_csv("/data/projects/punim0478/setiawand/bible-nmt/dictionary/freq_count_100_oov_dict.csv", index=False)

Total matches found: 253
Unique words found: 163

Selected top 100 unique words (with highest frequency counts)
Final result has 152 total entries for 100 unique words
Entries per word - Max: 8, Min: 1, Avg: 1.5
Words with multiple entries: 36
Top 5 words with most entries:


,entry_count
word,
kemarahannya,8
bercerita,4
sesudahmu,4
permukaan,3
berjanjilah,3
